# Genie BEFORE: Confirming the Structural Gap

**Run on serverless compute** (no Neo4j Spark Connector required).

This notebook asks five questions to the BEFORE Genie Space — the space pointed
at the base tables before graph enrichment. The first question is a tabular
warm-up that Genie answers correctly. The next three questions target structure
that lives in the transfer network topology, not in any row or column the base
tables carry. Each of those three misses is evidence of a structural gap in the
catalog, not a limitation of Genie.

The final question previews a business query that the enriched catalog will
answer in `05_genie_after`.

In [ ]:
%pip install --upgrade databricks-sdk --quiet
dbutils.library.restartPython()

In [ ]:
CATALOG  = "graph-enriched-lakehouse"
SCHEMA   = "graph-enriched-schema"
VOLUME   = "graph-enriched-volume"

SPACE_ID = dbutils.secrets.get("neo4j-graph-engineering", "genie_space_id_before")

In [ ]:
from databricks.sdk import WorkspaceClient
from demo_utils import (
    genie_caller,
    load_ground_truth,
    check_risk_score_precision,
    check_community_purity,
    check_ring_pair_fraction,
)

w = WorkspaceClient()
print("Connected to:", w.config.host)
ask_genie = genie_caller(w, SPACE_ID)

In [ ]:
_GT_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/ground_truth.json"
gt = load_ground_truth(_GT_PATH)
ring_lists = [r["account_ids"] for r in gt["rings"]]

print(
    f"Ground truth: {gt['summary']['total_rings']} rings, "
    f"{gt['summary']['total_fraud_accounts']:,} fraud accounts, "
    f"{gt['summary']['total_whale_accounts']:,} whales"
)

## Warm-Up — Tabular Baseline

The first question has a clear tabular answer: rank accounts by total spend.
Genie resolves this with a straightforward aggregation. The result confirms
Genie is working and establishes a baseline before the structural questions run.

In [ ]:
WARMUP_QUESTION = "What are the top 10 accounts by total amount spent across all merchants?"

print("[W] warm_up — asking...")
response_warmup = ask_genie(WARMUP_QUESTION)

if response_warmup["df"] is not None:
    print(f"    Rows: {len(response_warmup['df'])}")
    print(f"    SQL:  {response_warmup['sql']}")
    display(response_warmup["df"])
else:
    print(f"    Status: {response_warmup['status']}")
    print(f"    Text:   {response_warmup['text']}")

print()
print("[W] warm_up — TABULAR BASELINE")
print("    Confirms Genie answers basic tabular aggregations correctly.")

## Structural Questions

The next three questions target structure that lives in the transfer network
topology — hub positions, community membership, and shared-merchant similarity.
These properties are not encoded in any column the base tables carry. Each result
is labeled `STRUCTURAL GAP CONFIRMED` or `UNEXPECTED SIGNAL FOUND` based on how
well the response aligns with the known fraud-ring ground truth.

### Question 1 — Hub Detection

An analyst investigating fraud would ask which accounts sit at the center of a
money-movement network. On the base catalog, Genie has no `risk_score` column to
rank by — only raw transaction counts and amounts. The result surfaces
high-volume legitimate accounts alongside ring members with no signal to
separate them.

**Expected outcome:** low precision at top-20; the structural gap is confirmed.

In [ ]:
HUB_QUESTION = (
    "Are there accounts that seem to be the hub of a money movement "
    "network that are potentially fraudulent?"
)

print("[1] hub_detection — asking...")
response_hub = ask_genie(HUB_QUESTION)

if response_hub["df"] is not None:
    print(f"    Rows: {len(response_hub['df'])}")
    print(f"    SQL:  {response_hub['sql']}")
    display(response_hub["df"])

    result_hub = check_risk_score_precision(response_hub["df"], gt, topn=20)
    verdict_hub = "UNEXPECTED SIGNAL FOUND" if result_hub["passed"] else "STRUCTURAL GAP CONFIRMED"

    print()
    print(f"[1] hub_detection — {verdict_hub}")
    print(f"    Metric:  precision={result_hub['precision']:.2f}  (after-GDS criterion: > 0.70)")
    print(f"    Finding: {result_hub['true_positives']}/{result_hub['topn']} top-{result_hub['topn']} accounts are known fraud ring members")
else:
    verdict_hub = "STRUCTURAL GAP CONFIRMED"
    print(f"    Status: {response_hub['status']}")
    print(f"    Text:   {response_hub['text']}")
    print()
    print(f"[1] hub_detection — {verdict_hub}")
    print("    Genie returned no tabular result; the base catalog has no score column to rank by.")

### Question 2 — Community Structure

Fraud rings transfer money heavily among themselves, forming tight clusters in
the transfer graph. On the base catalog, Genie can only query rows and columns —
there is no `community_id` column, so any grouping over transfer pairs collapses
rings and legitimate accounts into the same result.

**Expected outcome:** low ring coverage; the structural gap is confirmed.

In [ ]:
COMMUNITY_QUESTION = "Find groups of accounts transferring money heavily among themselves."

print("[2] community_structure — asking...")
response_comm = ask_genie(COMMUNITY_QUESTION)

if response_comm["df"] is not None:
    print(f"    Rows: {len(response_comm['df'])}")
    print(f"    SQL:  {response_comm['sql']}")
    display(response_comm["df"])

    result_comm = check_community_purity(response_comm["df"], ring_lists)
    verdict_comm = "UNEXPECTED SIGNAL FOUND" if result_comm["passed"] else "STRUCTURAL GAP CONFIRMED"

    print()
    print(f"[2] community_structure — {verdict_comm}")
    print(f"    Metric:  max_ring_coverage={result_comm['max_ring_coverage']:.2f}  (after-GDS criterion: > 0.80)")
    print(f"    Finding: {result_comm['groups_returned']} group(s) returned; best covered {result_comm['max_ring_coverage']:.0%} of a single fraud ring")
else:
    verdict_comm = "STRUCTURAL GAP CONFIRMED"
    print(f"    Status: {response_comm['status']}")
    print(f"    Text:   {response_comm['text']}")
    print()
    print(f"[2] community_structure — {verdict_comm}")
    print("    Genie returned no tabular result; community_id does not exist in base tables.")

### Question 3 — Merchant Overlap

Fraud ring members share a small set of anchor merchants. On the base catalog,
ranking pairs by raw shared-merchant count surfaces high-volume legitimate
accounts that share many merchants by chance. The Jaccard normalization that
separates ring pairs from volume-inflated pairs does not exist as a column.

**Expected outcome:** low same-ring fraction; the structural gap is confirmed.

In [ ]:
MERCHANT_QUESTION = "Which pairs of accounts have visited the most merchants in common?"

print("[3] merchant_overlap — asking...")
response_merch = ask_genie(MERCHANT_QUESTION)

if response_merch["df"] is not None:
    print(f"    Rows: {len(response_merch['df'])}")
    print(f"    SQL:  {response_merch['sql']}")
    display(response_merch["df"])

    df_m = response_merch["df"]
    account_cols = [c for c in df_m.columns if "account" in c.lower()]
    if len(account_cols) >= 2:
        a_col, b_col = account_cols[0], account_cols[1]
        pairs = list(zip(df_m[a_col].astype(int), df_m[b_col].astype(int)))
        result_merch = check_ring_pair_fraction(pairs, ring_lists)
        verdict_merch = "UNEXPECTED SIGNAL FOUND" if result_merch["passed"] else "STRUCTURAL GAP CONFIRMED"

        print()
        print(f"[3] merchant_overlap — {verdict_merch}")
        print(f"    Metric:  same_ring_fraction={result_merch['same_ring_fraction']:.2f}  (after-GDS criterion: > 0.60 with >=5 pairs)")
        print(f"    Finding: {result_merch['same_ring_pairs']}/{result_merch['total_pairs']} top-similarity pairs are same-ring")
    else:
        verdict_merch = "STRUCTURAL GAP CONFIRMED"
        print()
        print(f"[3] merchant_overlap — {verdict_merch}")
        print(f"    Could not identify two account columns in result. Columns: {list(df_m.columns)}")
else:
    verdict_merch = "STRUCTURAL GAP CONFIRMED"
    print(f"    Status: {response_merch['status']}")
    print(f"    Text:   {response_merch['text']}")
    print()
    print(f"[3] merchant_overlap — {verdict_merch}")
    print("    Genie returned no tabular result; similarity_score does not exist in base tables.")

## Teaser — A Question the Enriched Catalog Can Answer

The following question asks for portfolio composition over structural segments:
what share of accounts sits in communities flagged as ring candidates, broken out
by region. The columns this question needs — `community_id` and
`is_ring_community` — do not exist in the base tables.

After the enrichment pipeline runs (`02_neo4j_ingest` → `03_aura_gds_guide` →
`04_pull_gold_tables`), Genie answers this question in `05_genie_after`.

In [ ]:
TEASER_QUESTION = (
    "What share of accounts sits in communities flagged as ring candidates, "
    "broken out by region?"
)

print("[T] teaser_portfolio — asking...")
response_teaser = ask_genie(TEASER_QUESTION)

print()
print("[T] teaser_portfolio — NOT AVAILABLE ON THIS CATALOG — answered in AFTER run")
print("    Note: community_id and is_ring_community columns do not exist in base tables.")
print()
if response_teaser["df"] is not None:
    print(f"    Rows: {len(response_teaser['df'])}")
    print(f"    SQL:  {response_teaser['sql']}")
    display(response_teaser["df"])
elif response_teaser["text"]:
    print(f"    Text: {response_teaser['text'][:300]}")
else:
    print(f"    Status: {response_teaser['status']}")

In [ ]:
print("=" * 78)
print("Genie BEFORE — Summary")
print("=" * 78)
print()
print("Purpose: confirm structural gap on base tables (misses are expected evidence)")
print()
print("  [W] warm_up              TABULAR BASELINE")
print(f"  [1] hub_detection        {verdict_hub}")
print(f"  [2] community_structure  {verdict_comm}")
print(f"  [3] merchant_overlap     {verdict_merch}")
print("  [T] teaser_portfolio     NOT AVAILABLE ON THIS CATALOG")
print()
print("-" * 78)
print("The base catalog cannot resolve structural questions from row-level SQL.")
print("Enrichment unlocks portfolio, cohort, community, operational, and merchant")
print("questions in the AFTER run.")

## Next Steps

Proceed to **`02_neo4j_ingest`** to load the five base tables into Neo4j as a
property graph. After the full pipeline (`02_neo4j_ingest` → `03_aura_gds_guide`
→ `04_pull_gold_tables`), open `05_genie_after` to ask analyst questions against
the enriched catalog.